### Computing Cyclomatic complexity from generated .dot file

In [9]:
import os
import re
import pandas as pd

data = []

# Iterate through all .dot files in current directory
for filename in os.listdir('./dots'):
        with open(f"./dots/{filename}", 'r') as f:
            text = f.read()

        # Extract nodes and edges using regex
        nodes = re.findall(r'\b(fn_\d+_basic_block_\d+)\b\s*\[', text)
        edges = re.findall(r'(fn_\d+_basic_block_\d+):[a-z]+ -> (fn_\d+_basic_block_\d+):[a-z]+', text)

        nodes = sorted(set(nodes))
        N = len(nodes)
        E = len(edges)

        # Cyclomatic Complexity formula: CC = E - N + 2
        CC = E - N + 2

        # Extract program name
        program_name = os.path.splitext(filename)[0].split(".016t.cfg")[0].replace("a-","")

        data.append({
            "Program Name": program_name,
            "No. of Nodes (N)": N,
            "No. of Edges (E)": E,
            "Cyclomatic Complexity (CC)": CC
        })
# Create dataframe
df = pd.DataFrame(data)

In [10]:
df_sorted = df.sort_values('Cyclomatic Complexity (CC)').reset_index(drop=True)
df_sorted['Program No.'] = df_sorted.index + 1
df_sorted[["Program No."]+list(df_sorted.columns[:4])]

,Program No.,Program Name,No. of Nodes (N),No. of Edges (E),Cyclomatic Complexity (CC)
0,1,new_test,7,8,3
1,2,scheduler_rr,45,63,20
2,3,adventure,84,124,42
3,4,2048,229,331,104


### Reaching Definitions Analysis:

#### Parsing logic
| Component         | Meaning                      | Example Match              |   
| ----------------- | ---------------------------- | -------------------------- |
| `<bb\s+(\d+)>`    | Captures block number        | `<bb 2>` → `"2"`           |   
| `\s*:`            | Optional space + colon       | `<bb 2> :`                 |   
| `(.*?)`           | Captures block content       | `x = 0; y = 5;`            |   
| `(?=(?:<bb\s+\d+\s*:> \| $))`|                         | Stops at next block or end | 
| `re.S`            | Allows `.` to match newlines | —                          |   

In [ ]:
import re
import os
import pandas as pd

def read_cfg(filename):
    """Read .cfg file"""
    with open(filename, 'r') as f:
        return f.read()

def parse_blocks_and_stmts(text):
    """Extract basic blocks and their statements"""
    blocks_raw = re.findall(r"<bb\s+(\d+)>\s*:(.*?)(?=(?:<bb\s+\d+>\s*:|$))", text, re.S)
    # basic block to statements mapping
    bb_to_stmts = {}
    blocks = []
    for bid, body in blocks_raw:
        stmts = [s.strip() for s in body.splitlines() if s.strip()]
        bb_to_stmts[bid] = stmts
        blocks.append(bid)
    return sorted(set(blocks), key=lambda x: int(x)), bb_to_stmts

def parse_succ_map(text):
    """Parse lines like '2 succs { 3 6 }' to identify edges"""
    succ_pattern = re.compile(r"(\d+)\s+succs\s+\{\s*([0-9\s]*)\}")
    succ_map = {}
    for src, dsts in succ_pattern.findall(text):
        succs = re.findall(r"\d+", dsts)
        succ_map[src] = succs
    return succ_map

from collections import defaultdict
def extract_definitions(bb_to_stmts):
    """Identify definitions and assign D1, D2, ..."""
    lhs_pattern = re.compile(r"\b([A-Za-z_][A-Za-z0-9_]*)\s*=")
    definitions = []
    defs_by_block = defaultdict(list)
    var_to_defs = defaultdict(list)
    def_id = 1

    for b, stmts in bb_to_stmts.items():
        for s in stmts:
            m = lhs_pattern.search(s)
            if m:
                var = m.group(1)
                definitions.append((def_id, var, b, s))
                defs_by_block[b].append(def_id)
                var_to_defs[var].append(def_id)
                def_id += 1
    
    return definitions, defs_by_block, var_to_defs

def build_gen_kill(blocks, var_to_defs, definitions):
    """Builds gen and kill for each block"""
    gen, kill = {}, {}

    # Map from definition ID to (def_id, var, block, stmt)
    defs_to_var = {d[0]: d for d in definitions}

    # Precompute last definition of each variable in each block
    last_def_in_block = {}
    for d_id, var, b, stmt in definitions:
        last_def_in_block[(b, var)] = d_id

    for b in blocks:
        # Only keep last definition of each variable in the block
        gen[b] = set(d_id for (blk, var), d_id in last_def_in_block.items() if blk == b)
        kill[b] = set()

        # For each definition in gen[b], kill all other definitions of same variable in other blocks
        for d_id in gen[b]:
            _, var, blk, _ = defs_to_var[d_id]
            # Identifying in which other blocks this variable is redefined
            for other_did in var_to_defs[var]:
                if other_did != d_id and defs_to_var[other_did][2] != b:
                    kill[b].add(other_did)

    return gen, kill

def build_pred_map(blocks, succ_map):
    """Builds predecessor map"""
    pred = {b: [] for b in blocks}
    for src, dsts in succ_map.items():
        for d in dsts:
            pred.setdefault(d, []).append(src)
    return pred

def make_dataframe(snapshot, gen, kill, iteration):
    """Create a pandas DataFrame for a given iteration"""
    rows = []
    for b in sorted(snapshot.keys(), key=lambda x: int(x)):
        rows.append({
            "Iteration": iteration,
            "Basic Block": b,
            "gen[B]": sorted(gen[b]),
            "kill[B]": sorted(kill[b]),
            "in[B]": sorted(snapshot[b][0]),
            "out[B]": sorted(snapshot[b][1]),
        })
    return pd.DataFrame(rows)

def reaching_defs(blocks, pred, gen, kill):
    """Iterative reaching definition analysis"""
    in_sets = {b: set() for b in blocks}
    out_sets = {b: set() for b in blocks}
    history = []
    changed = True
    iteration = 0

    while changed:
        changed = False
        iteration += 1
        snapshot = {}
        for b in blocks:
            # Step 1: in[B] = ⋃ out[P] for all predecessors P
            new_in = set().union(*[out_sets[p] for p in pred.get(b, [])])
            
            # Step 2: out[B] = gen[B] ∪ (in[B] - kill[B])
            new_out = gen[b] | (new_in - kill[b])
            
            # Step 3: check for changes
            if new_in != in_sets[b] or new_out != out_sets[b]:
                changed = True
            
            in_sets[b], out_sets[b] = new_in, new_out
            snapshot[b] = (new_in.copy(), new_out.copy())
        
        print(make_dataframe(snapshot, gen, kill, iteration).to_string(index=False))
        history.append(snapshot)
    
    return in_sets, out_sets, history

In [20]:
for filename in os.listdir('./cfgs'):
    if filename.endswith('.cfg'):
        # Reading .cfg file
        fname = filename.replace("a-","").replace(".016t.cfg","")
        print(f"\n\nAnalysis for {fname}")
        text = read_cfg(f"./cfgs/{filename}")
        blocks, bb_to_stmts = parse_blocks_and_stmts(text)
        succ_map = parse_succ_map(text)
        definitions, defs_by_block, var_to_defs = extract_definitions(bb_to_stmts)
        print("\n=== Definition Mapping ===")
        mapping_df = pd.DataFrame(definitions, columns=["DefID", "Variable", "Block", "Statement"])
        mapping_df["DefID"] = mapping_df["DefID"].apply(lambda x: f"D{x}")
        mapping_df["Statement"] = mapping_df["Statement"].apply(lambda s: s.replace(f"c_files/{fname.replace("cfg","c")}:","Line no:")) 
        print(mapping_df.to_string(index=False))
        gen, kill = build_gen_kill(blocks, var_to_defs, definitions)
        pred = build_pred_map(blocks,succ_map)
        print("\n === Iteration-wise Table ===")
        in_sets, out_sets, history = reaching_defs(blocks,pred,gen,kill)
        print("\n=== Final (Converged) in[B] and out[B] ===")
        final_df = pd.DataFrame({
                "Block": blocks,
                "gen[B]": [sorted(gen[b]) for b in blocks],
                "kill[B]": [sorted(kill[b]) for b in blocks],
                "in[B]": [sorted(in_sets[b]) for b in blocks],
                "out[B]": [sorted(out_sets[b]) for b in blocks]
        })
        print(final_df.to_string(index=False))
        # Identify variables with multiple reaching definitions
        print("\n=== Variables with Multiple Reaching Definitions ===")
        def_map = {d[0]: d[1] for d in definitions}  # defID to variable
        for b in blocks:
            in_defs = [def_map[d] for d in in_sets[b]]
            mult = {v for v in in_defs if in_defs.count(v) > 1}
            if mult:
                if len(mult)>1:
                    print(f"Block {b}: variable(s) {', '.join(mult)} have multiple reaching definitions")
                else:
                    print(f"Block {b}: variable(s) {', '.join(mult)} has multiple reaching definitions")



Analysis for 2048.cfg

=== Definition Mapping ===
DefID        Variable Block                                                                                    Statement
   D1              _1     2                                                               [Line no:9:11] _1 = time (0B);
   D2              _2     2                                              [Line no:9:5 discrim 1] _2 = (unsigned int) _1;
   D3      playGround     2                                                              [Line no:12:9] playGround = {};
   D4              _3     3                                                   [Line no:28:12 discrim 2] _3 = getchar ();
   D5              _4     4                                                                [Line no:32:17] _4 = rand ();
   D6           coord     4                                                    [Line no:32:9 discrim 1] coord = _4 % 16;
   D7               x     4                                                                [Line no:3